# Notebook 01: Data Ingestion, Schema Auditing & Cleaning Pipeline
## University Student Performance Dataset

---

### Objectives
1. Ingest raw academic survey and grade records from `data/raw/students_raw.csv`.
2. Perform comprehensive data quality audit (missing values, extreme outliers, invalid ranges).
3. Enforce deterministic canonical validation rules **V1 through V7**:
   - **V1:** Unique primary identifier verification (`student_id`).
   - **V2:** Attendance percentage bounded strictly within $[0, 100]$.
   - **V3:** Weekly self-study hours validated as non-negative ($\ge 0$).
   - **V4:** Assignment average bounded within $[0, 100]$.
   - **V5:** Midterm exam marks bounded within $[0, 100]$.
   - **V6:** Prior cumulative GPA bounded within $[0.0, 4.0]$.
   - **V7:** Target final examination score bounded within $[0, 100]$.
4. Apply skewness-aware missing value imputation (median for skewed, mean for normal).
5. Export verified canonical dataset to `data/interim/students_clean.csv`.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import yaml

# Load project configuration
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration paths:")
for k, v in config['paths'].items():
    print(f"  {k:10s} : {v}")

Configuration paths:
  raw        : data/raw/students_raw.csv
  data_source : data/raw/DATA_SOURCE.md
  data_dictionary : data/raw/DATA_DICTIONARY.md
  interim    : data/interim/students_clean.csv
  train      : data/processed/train.csv
  test       : data/processed/test.csv
  figures    : reports/figures
  tables     : reports/tables
  models     : models


## 1. Raw Data Ingestion & Initial Audit

In [2]:
raw_path = '../' + config['paths']['raw']
df_raw = pd.read_csv(raw_path)

print(f"Raw dataset shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns\n")
print("First 5 records:")
df_raw.head()

Raw dataset shape: 402 rows, 7 columns

First 5 records:


,student_id,attendance_pct,study_hours_week,assignment_avg,midterm_score,previous_gpa,final_score
0,STU_0001,73.4,26.2,90.1,76.9,3.39,82.9
1,STU_0002,73.0,18.3,82.8,73.3,3.04,66.4
2,STU_0003,84.5,22.4,100.0,83.1,3.50,88.6
3,STU_0004,93.5,27.1,100.0,96.7,3.53,97.1
4,STU_0005,73.0,24.0,77.2,77.0,NaN,73.0


In [3]:
print("Raw Dataset Info:")
df_raw.info()

print("\nMissing Values Count:")
missing_summary = pd.DataFrame({
    'Missing Count': df_raw.isnull().sum(),
    'Missing Pct (%)': (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
})
missing_summary[missing_summary['Missing Count'] > 0]

Raw Dataset Info:


<class 'pandas.DataFrame'>
RangeIndex: 402 entries, 0 to 401
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   student_id        402 non-null    str    
 1   attendance_pct    402 non-null    float64
 2   study_hours_week  394 non-null    float64
 3   assignment_avg    396 non-null    float64
 4   midterm_score     402 non-null    float64
 5   previous_gpa      397 non-null    float64
 6   final_score       400 non-null    float64
dtypes: float64(6), str(1)
memory usage: 25.3 KB

Missing Values Count:


,Missing Count,Missing Pct (%)
study_hours_week,8,1.99
assignment_avg,6,1.49
previous_gpa,5,1.24
final_score,2,0.50


## 2. Enforcement of Validation Rules (V1 - V7)
We clean invalid entries, remove duplicates, and log all modifications.

In [4]:
from src.cleaning import validate_and_clean_data

df_clean, df_log, df_missing = validate_and_clean_data(
    raw_path=raw_path,
    interim_path='../' + config['paths']['interim'],
    log_path='../reports/tables/data_cleaning_log.csv',
    missing_table_path='../reports/tables/missing_values_report.csv'
)

print(f"Cleaned dataset rows: {len(df_clean)}")
print(f"Total cleaning actions logged: {len(df_log)}")
print("\nCleaning Log Sample:")
df_log.head(10)

Cleaning complete. Retained 398 / 402 rows.
Clean data saved to ../data/interim/students_clean.csv
Cleaning log saved to ../reports/tables/data_cleaning_log.csv
Missing values report saved to ../reports/tables/missing_values_report.csv
Cleaned dataset rows: 398
Total cleaning actions logged: 12

Cleaning Log Sample:


,rule_id,action,rows_affected,details
0,V1,Checked required canonical columns,0,All columns present
1,V7,Standardized student_id format,402,Trimmed whitespace
2,V2,Deduplicated on student_id,2,Removed 2 duplicate records
3,V4,Coerced numeric columns to float,400,Converted bad characters to NaN
4,V3,Checked numeric range boundaries,1,Set 1 invalid entries to NaN
5,V5,Dropped records missing target (final_score),2,Removed 2 rows
6,V6,Dropped rows missing > 50% of features,0,Removed 0 rows
7,A4,Imputed attendance_pct using mean,1,Filled with value: 77.85
8,A4,Imputed study_hours_week using mean,8,Filled with value: 21.3
9,A4,Imputed assignment_avg using median,6,Filled with value: 85.9


## 3. Post-Cleaning Verification
We verify that all feature columns conform strictly to their canonical mathematical bounds.

In [5]:
checks = [
    ("V1: Duplicate student_id count", df_clean['student_id'].duplicated().sum() == 0),
    ("V2: Attendance within [0, 100]", df_clean['attendance_pct'].between(0, 100).all()),
    ("V3: Study hours non-negative", (df_clean['study_hours_week'] >= 0).all()),
    ("V4: Assignment average within [0, 100]", df_clean['assignment_avg'].between(0, 100).all()),
    ("V5: Midterm score within [0, 100]", df_clean['midterm_score'].between(0, 100).all()),
    ("V6: Prior GPA within [0.0, 4.0]", df_clean['previous_gpa'].between(0.0, 4.0).all()),
    ("V7: Final score within [0, 100]", df_clean['final_score'].between(0, 100).all()),
    ("Zero remaining nulls", df_clean.isnull().sum().sum() == 0)
]

for check_name, passed in checks:
    status = "PASSED" if passed else "FAILED"
    print(f"[{status}] {check_name}")

[PASSED] V1: Duplicate student_id count
[PASSED] V2: Attendance within [0, 100]
[PASSED] V3: Study hours non-negative
[PASSED] V4: Assignment average within [0, 100]
[PASSED] V5: Midterm score within [0, 100]
[PASSED] V6: Prior GPA within [0.0, 4.0]
[PASSED] V7: Final score within [0, 100]
[PASSED] Zero remaining nulls


In [6]:
print("Cleaned Dataset Summary Statistics:")
df_clean.describe().round(2)

Cleaned Dataset Summary Statistics:


,attendance_pct,study_hours_week,assignment_avg,midterm_score,previous_gpa,final_score
count,398.00,398.00,398.00,398.00,398.00,398.00
mean,77.85,21.30,85.06,79.21,3.01,78.11
std,10.40,5.49,11.07,13.71,0.48,11.96
min,45.40,5.80,42.10,25.50,1.80,42.50
25%,70.18,17.60,77.45,69.40,2.68,69.53
50%,78.25,21.20,85.90,79.85,3.02,79.40
75%,85.10,24.78,94.20,89.88,3.33,87.05
max,100.00,39.30,100.00,100.00,4.00,100.00
